In [1]:
from datasets import load_dataset, concatenate_datasets

dataset_easy = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets")
dataset_easy_v2 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets_v2")
dataset_easy_v3 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets_v3")

# dataset_hard = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "hard_triplets")
# dataset_hard_v2 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "hard_triplets_v2")

ds = concatenate_datasets(
    [
        dataset_easy["train"],
        dataset_easy_v2["train"],
        dataset_easy_v3["train"],
        # dataset_hard["train"],
        # dataset_hard_v2["train"],
    ]
)

ds


Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 5655
})

In [2]:
ds_triplets = ds
def to_pairs(batch):
    pairs = []
    for a, p, n in zip(batch["anchor"], batch["positive"], batch["negative"]):
        pairs.append({"text1": a, "text2": p, "label": 1.0})
        pairs.append({"text1": a, "text2": n, "label": 0.0})
    return {"text1": [x["text1"] for x in pairs],
            "text2": [x["text2"] for x in pairs],
            "label": [x["label"] for x in pairs]}

ds_pairs = ds_triplets.map(to_pairs, batched=True, remove_columns=ds_triplets.column_names)
ds_pairs

Map:   0%|          | 0/5655 [00:00<?, ? examples/s]

Dataset({
    features: ['text1', 'text2', 'label'],
    num_rows: 11310
})

In [3]:
ds_pairs[0]

{'text1': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'text2': 'The Audit Associate/Audit Assistant Associate performs designated audit procedures under guidance, gradually understanding the rationale behind assigned tasks within the audit framework. This role involves adapti

In [4]:
from sentence_transformers import CrossEncoder, InputExample, losses
import torch
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from torch.utils.data import DataLoader

model = CrossEncoder("answerdotai/ModernBERT-base", num_labels=1)  # score in [0,1] via sigmoid
loss = BinaryCrossEntropyLoss(model=model, pos_weight=torch.tensor(5))


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments

args = CrossEncoderTrainingArguments(
    output_dir="models/reranker-run",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,
    # load_best_model_at_end=True,
    # metric_for_best_model="eval_ndcg@10",
    # eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
)
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=ds_pairs,
    loss=loss,
    # evaluator=nano,
)
trainer.train()


wandb: Currently logged in as: dnth to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=354, training_loss=0.16260874473442466, metrics={'train_runtime': 202.796, 'train_samples_per_second': 55.77, 'train_steps_per_second': 1.746, 'total_flos': 0.0, 'train_loss': 0.16260874473442466, 'epoch': 1.0})

In [6]:
trainer.save_model()
trainer.model.push_to_hub("dnth/ssf-reranker-modernbert-embed-base-v0", exist_ok=True)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmp8a3024c7/model.safetensors    :   0%|          |  325kB /  598MB            

'https://huggingface.co/dnth/ssf-reranker-modernbert-embed-base-v0/commit/e94e16e359304641fd8611eed86deefe4a45efe8'